# Check Dataset class 

In [1]:
import os
import pandas as pd
from tqdm import tqdm
if os.getcwd().endswith("notebooks"):
    os.chdir("../")

from src.supervised_TCN.dataset import MotionFeatureDataset

In [2]:
processed_path = "data/processed"
paths = [os.path.join(processed_path, f) for f in os.listdir(processed_path) if f.endswith("10fps_processed.pkl")]

df_dict = {}
cols_to_keep = ['frame', 'hand_label', 'cx_smooth', 'cy_smooth']

# load df with where first column in csv serves as index
df_vid_name_map = pd.read_csv("data/scores/vid_name_map.csv", index_col=0)

with tqdm(total=len(paths), desc="Loading processed data") as pbar:
    for path in paths:
        vid = os.path.basename(path).replace("_10fps_processed.pkl", "")
        vid = vid.replace("hand_tracking_", "")
        participant_id = df_vid_name_map.loc[vid]['Participant Number']
        if int(participant_id) == 8:
            continue
        df_dict[(vid, int(participant_id))] = pd.read_pickle(path)[cols_to_keep]
        pbar.update(1)

Loading processed data:  97%|█████████▋| 83/86 [00:12<00:00,  6.85it/s]


In [3]:
df_scores = pd.read_csv("data/scores/merged_scores.csv")[['Vid_Name', 'QRS_Overal']]
grs_scores = df_scores.set_index('Vid_Name')['QRS_Overal'].to_dict()
grs_scores

{'2024-01-15_13-18-23': 48.5,
 '2024-01-15_13-37-36': 45.0,
 '2024-01-15_14-03-23': 60.5,
 '2024-01-15_14-32-45': 39.25,
 '2024-01-15_15-05-31': 38.0,
 '2024-01-15_15-38-13': 47.0,
 '2024-01-15_15-58-44': 62.0,
 '2024-01-15_16-17-19': 64.5,
 '2024-01-15_16-31-26': 63.0,
 '2024-01-15_17-37-24': 37.25,
 '2024-01-15_17-57-25': 40.5,
 '2024-01-15_18-17-18': 42.0,
 '2024-01-16_14-30-29': 36.0,
 '2024-01-16_15-20-49': 47.25,
 '2024-01-16_15-41-03': 43.0,
 '2024-01-16_16-03-11': 35.0,
 '2024-01-16_16-34-19': 44.25,
 '2024-01-16_17-00-33': 41.5,
 '2024-01-17_16-04-01': 44.0,
 '2024-01-17_16-22-28': 47.0,
 '2024-01-17_16-48-38': 41.25,
 '2024-01-18_10-40-40': 58.66666666666666,
 '2024-01-18_10-52-25': 62.66666666666666,
 '2024-01-18_11-08-40': 60.0,
 '2024-01-18_13-11-31': 60.75,
 '2024-01-18_13-25-00': 55.25,
 '2024-01-18_13-45-41': 62.25,
 '2024-01-18_14-39-24': 47.0,
 '2024-01-18_14-55-56': 52.25,
 '2024-01-18_15-17-27': 47.5,
 '2024-01-18_16-55-29': 59.75,
 '2024-01-18_17-08-23': 66.0,
 '20

# Single Training Split

In [4]:
from src.supervised_TCN.tcn import SurgicalTCN
from src.supervised_TCN.train import train_tcn, run_losocv
from torch.utils.data import DataLoader, Subset
import numpy as np

In [5]:
validation_goups = [1, 11, 17, 18, 27, 28]
train_groups = [g for g in range(1, 30) if (g not in validation_goups or g==8)]

train_df_dict = {key: df_dict[key] for key in df_dict.keys() if key[1] in train_groups}

training_stats = MotionFeatureDataset.compute_scaling_stats(
    df_dict=train_df_dict, 
    hand="Right", 
    orig_fps=30.0,
    max_gap_seconds=0.11
)


In [6]:
dataset = MotionFeatureDataset(df_dict, grs_scores, hand="Right", window_size=100, step_size=25, orig_fps=30.0, max_gap_seconds=0.11, device="cpu", scaling_stats=training_stats)

Processing Videos: 100%|██████████| 83/83 [00:05<00:00, 15.01it/s]


In [7]:
groups = np.array([item[3] for item in dataset.index_map])

train_indices = [i for i, g in enumerate(groups) if g in train_groups]
val_indices = [i for i, g in enumerate(groups) if g in validation_goups]

train_subset = Subset(dataset, train_indices)
val_subset = Subset(dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

In [8]:
model = SurgicalTCN(num_inputs=7, num_channels=[32, 32, 32], kernel_size=3, dropout=0.4)

In [10]:
best_model, best_val_corr = train_tcn(model, train_loader, val_loader, device="cpu")

RuntimeError: Given groups=1, weight of size [32, 7, 3], expected input[32, 99, 7] to have 7 channels, but got 99 channels instead